# perplexityai smoke test

Verifies that the `perplexityai` distribution (import name: `perplexity`) installs cleanly,
all submodules import, and the primary client / resource / type surfaces are constructible.

No network calls are made: the API key is a dummy value and we only test offline construction.

In [ ]:
# Example 1: top-level public API surface is exposed.
import perplexity

# Stainless-generated SDKs expose a sync + async client class.
expected_names = ["Perplexity", "AsyncPerplexity"]
present = [n for n in expected_names if hasattr(perplexity, n)]
print(f"Exposed clients: {present}")

# At minimum the sync client must exist.
assert "Perplexity" in present, f"missing sync client; perplexity dir = {dir(perplexity)[:20]}"

# Error hierarchy: stainless SDKs expose a PerplexityError or APIError base.
error_names = [n for n in dir(perplexity) if n.endswith("Error")]
print(f"Exposed error types: {error_names[:10]}")
assert error_names, "expected at least one *Error class on the top-level module"

In [ ]:
# Example 2: sync client constructs with a dummy api key (no network).
from perplexity import Perplexity

client = Perplexity(api_key="sk-dummy-not-real-key-for-offline-test")
print(f"type(client) = {type(client).__name__}")
print(f"base_url = {client.base_url}")

# Stainless clients expose .with_options() for per-call overrides.
assert hasattr(client, "with_options"), "client missing .with_options()"

# And a .close() / context manager.
assert hasattr(client, "close"), "client missing .close()"
client.close()
print("client closed cleanly")

In [ ]:
# Example 3: async client constructs symmetrically.
from perplexity import AsyncPerplexity

aclient = AsyncPerplexity(api_key="sk-dummy-not-real-key-for-offline-test")
print(f"type(aclient) = {type(aclient).__name__}")
print(f"base_url = {aclient.base_url}")

# Async client also has .with_options() and .close() (coroutine).
assert hasattr(aclient, "with_options"), "async client missing .with_options()"
assert hasattr(aclient, "close"), "async client missing .close()"

# Don't actually await close() here -- just verify the attribute exists.
print("async client constructed; close() attribute present")

In [ ]:
# Example 4: resource namespaces are attached to the client.
# Perplexity's SDK is API-surface-stable: chat completions + search are the headline resources.
from perplexity import Perplexity

client = Perplexity(api_key="sk-dummy")

# Probe for known resource namespaces. At least one must be present.
candidate_resources = ["chat", "search", "async_chat", "completions"]
found = {name: type(getattr(client, name)).__name__ for name in candidate_resources if hasattr(client, name)}
print(f"Resource namespaces found on client: {found}")

assert found, f"no recognized resource namespaces on client; dir = {[n for n in dir(client) if not n.startswith('_')][:20]}"
client.close()

In [ ]:
# Example 5: error class hierarchy is well-formed.
import perplexity

# Common stainless-style errors.
error_candidates = [
    "APIError",
    "APIConnectionError",
    "APITimeoutError",
    "AuthenticationError",
    "BadRequestError",
    "RateLimitError",
    "InternalServerError",
]
found_errors = {name: getattr(perplexity, name) for name in error_candidates if hasattr(perplexity, name)}
print(f"Found {len(found_errors)} error classes: {list(found_errors)}")

# Every found error must be an Exception subclass.
for name, cls in found_errors.items():
    assert isinstance(cls, type), f"{name} is not a class"
    assert issubclass(cls, Exception), f"{name} is not an Exception subclass"

# Sanity: at least one error type must exist on the public surface.
assert found_errors, "no error classes found on perplexity module"

In [ ]:
# Example 6: every submodule imports.
import importlib
import pkgutil

import perplexity

submodules = [m.name for m in pkgutil.walk_packages(perplexity.__path__, "perplexity.")]
failures = {}
for name in submodules:
    try:
        importlib.import_module(name)
    except Exception as e:
        failures[name] = f"{type(e).__name__}: {e}"

print(f"Imported {len(submodules) - len(failures)} of {len(submodules)} submodules")
assert submodules, "pkgutil found no submodules under perplexity"
assert "perplexity.resources" in submodules and "perplexity.types" in submodules, submodules[:20]
assert not failures, failures
